# Baseline direction and unit-corrected ordering

| Cell | Output | Thesis |
|---|---|---|
| 2 | H1/H3/H5 band = 0.0145 | §4.3, Table 4.2 row 1b |
| 3 | mean full 95% CI = 0.0099 | §4.5, Table 4.2 row 4 |
| 4 | NC spread: median 2, 74% ≥2, Jul-2018 = 1–4 | §4.3, abstract |
| 5 | above/below by year, 15 scenarios | §4.7 |
| 6 | above/below, 9 high-coverage only | §4.7 |
| 7 | argmin/argmax per metric (2017 peak) | §4.2 |

In [1]:
import pandas as pd
df = pd.read_parquet("../data/derived/sensitivity_full.parquet")
hi = df[df.window >= pd.Timestamp("2018-01-01")]

nc = (hi[hi.metric == "nakamoto"]
      .pivot_table(index="window", columns=["heuristic", "treatment"], values="value"))
rng = nc.max(axis=1) - nc.min(axis=1)
print("median spread:", rng.median(), "| max:", rng.max())
print("months where NC spans >= 2:", f"{(rng >= 2).mean():.0%}")
worst = nc.loc[rng.idxmax()]
print(f"worst month {rng.idxmax():%Y-%m}: NC ranges {worst.min():.0f} to {worst.max():.0f}")

median spread: 2.0 | max: 3.0
months where NC spans >= 2: 74%
worst month 2018-07: NC ranges 1 to 4


In [2]:
for metric in ["hhi", "nakamoto", "cr3", "gini", "shannon"]:
    piv = (hi[hi.metric == metric]
           .pivot_table(index="window", columns=["heuristic", "treatment"], values="value"))
    base = piv[("H1", "U2")]
    pct = piv.rank(axis=1, pct=True)[("H1", "U2")].mean()
    gap = (base - piv.median(axis=1)).mean()
    print(f"{metric:9s} baseline percentile {pct:.2f}   signed gap vs median {gap:+.4f}")

hhi       baseline percentile 0.48   signed gap vs median -0.0021
nakamoto  baseline percentile 0.68   signed gap vs median +0.0882
cr3       baseline percentile 0.49   signed gap vs median -0.0049
gini      baseline percentile 0.40   signed gap vs median -0.0179
shannon   baseline percentile 0.56   signed gap vs median +0.0070


In [3]:
h1 = hi[(hi.heuristic == "H1") & (hi.metric == "hhi")].pivot_table(
        index="window", columns="treatment", values="value")
print(h1.mean())

treatment
U1    0.143097
U2    0.158948
U3    0.180040
dtype: float64


In [4]:
nc_all = hi[hi.metric=="nakamoto"].pivot_table(index="window", columns=["heuristic","treatment"], values="value")
pct_series = nc_all.rank(axis=1, pct=True)[("H1","U2")]
print(pct_series.groupby(pct_series.index.year).mean().round(2))

window
2018    0.74
2019    0.76
2020    0.76
2021    0.64
2022    0.62
2023    0.70
2024    0.71
2025    0.57
2026    0.52
Name: (H1, U2), dtype: float64


In [5]:
nc_all = hi[hi.metric=="nakamoto"].pivot_table(index="window", columns=["heuristic","treatment"], values="value")
base = nc_all[("H1","U2")]
above = nc_all.gt(base, axis=0).sum(axis=1)   # scenarios strictly MORE decentralised
below = nc_all.lt(base, axis=0).sum(axis=1)   # scenarios strictly LESS decentralised
ties  = 15 - above - below
out = pd.DataFrame({"above":above, "below":below, "ties":ties})
print(out.groupby(out.index.year).mean().round(1))

        above  below  ties
window                    
2018      0.0    6.2   8.8
2019      0.5    7.2   7.2
2020      0.0    6.8   8.2
2021      2.7    6.0   6.3
2022      2.9    5.4   6.7
2023      0.0    5.0  10.0
2024      0.0    5.2   9.8
2025      4.4    5.7   4.9
2026      6.3    6.0   2.7


In [6]:
# 1. Confirm the contrast — does HHI show the symmetric pattern I expect?
h = hi[hi.metric=="hhi"].pivot_table(index="window", columns=["heuristic","treatment"], values="value")
b = h[("H1","U2")]
print(pd.DataFrame({"above": h.gt(b,axis=0).sum(1), "below": h.lt(b,axis=0).sum(1)}
     ).groupby(h.index.year).mean().round(1))

# 2. Is the 2025-26 symmetry a coverage effect or an NC floor effect?
print(nc_all[("H1","U2")].groupby(nc_all.index.year).mean().round(2))

        above  below
window              
2018      7.8    6.2
2019      9.0    4.7
2020      7.8    6.2
2021      7.3    6.7
2022      7.7    6.3
2023      8.5    5.5
2024      7.0    7.0
2025      7.0    7.0
2026      7.0    7.0
window
2018    3.50
2019    3.92
2020    3.92
2021    3.58
2022    3.00
2023    2.00
2024    2.00
2025    2.00
2026    2.00
Name: (H1, U2), dtype: float64


C:\Users\Nakis\AppData\Local\Temp\ipykernel_8492\2126165492.py:4: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  print(pd.DataFrame({"above": h.gt(b,axis=0).sum(1), "below": h.lt(b,axis=0).sum(1)}
C:\Users\Nakis\AppData\Local\Temp\ipykernel_8492\2126165492.py:4: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  print(pd.DataFrame({"above": h.gt(b,axis=0).sum(1), "below": h.lt(b,axis=0).sum(1)}


In [7]:
below_mask = nc_all.lt(base, axis=0)
print(below_mask.mean().sort_values(ascending=False))

heuristic  treatment
H2         U3           1.000000
H4         U3           1.000000
           U1           1.000000
H2         U1           0.941176
H4         U2           0.921569
H2         U2           0.676471
H1         U3           0.225490
H5         U3           0.088235
H3         U3           0.088235
H1         U1           0.000000
           U2           0.000000
H3         U1           0.000000
           U2           0.000000
H5         U1           0.000000
           U2           0.000000
dtype: float64


In [10]:
cred = nc_all[[c for c in nc_all.columns if c[0] in ("H1","H3","H5")]]
b = cred[("H1","U2")]
print(pd.DataFrame({"above": cred.gt(b, axis=0).sum(axis=1),
                    "below": cred.lt(b, axis=0).sum(axis=1)}
     ).groupby(cred.index.year).mean().round(1))
print(cov[cov.window >= "2024-01-01"].groupby(...).mean())   # monthly tag coverage 2024–26

        above  below
window              
2018      0.0    0.3
2019      0.5    1.4
2020      0.0    0.8
2021      2.7    0.6
2022      2.9    0.3
2023      0.0    0.0
2024      0.0    0.0
2025      4.4    0.0
2026      6.3    0.0


NameError: name 'cov' is not defined

NameError: name 'cov' is not defined

In [11]:
# 1. Confirm the mechanism: which scenarios rise above in 2025-26?
above_mask = cred.gt(b, axis=0)
print(above_mask[above_mask.index >= "2025-01-01"].mean().sort_values(ascending=False))

# 2. Coverage by month, 2024 onwards -- `cov` didn't exist; rebuild it
#    from the block-level frame you used for Figure 4.1:
m = raw.assign(known=raw["pool"] != "unknown")
m["month"] = pd.to_datetime(m["timestamp"]).dt.to_period("M")
print(m.groupby("month")["known"].mean().loc["2024":].round(3))

heuristic  treatment
H1         U1           0.833333
H3         U1           0.833333
H5         U1           0.833333
           U2           0.666667
H3         U2           0.666667
           U3           0.611111
H5         U3           0.611111
H1         U2           0.000000
           U3           0.000000
dtype: float64


NameError: name 'raw' is not defined

In [2]:
import json, glob, re

for path in glob.glob("**/*.ipynb", recursive=True):
    nb = json.load(open(path, encoding="utf-8"))
    for i, cell in enumerate(nb.get("cells", [])):
        src = "".join(cell.get("source", []))
        if re.search(r"chance|toler|n_break|corrob|0\.13", src, re.I):
            print(f"===== {path}  [cell {i}] =====")
            print(src)
            for out in cell.get("outputs", []):
                txt = "".join(out.get("text", []))
                if txt:
                    print("---- output ----")
                    print(txt[:600])
            print()

===== changepoint_robustness.ipynb  [cell 0] =====
# RQ4 - robustness of the structural story (the capstone)

**Does attribution choice change the structural story of Bitcoin's mining
decentralisation?** Tested across all 15 attribution scenarios (H1-H5 x U1-U3).

**Why not raw change-point dates.** The decentralisation signal drifts *smoothly* (no
sharp piecewise steps -- the `gain%` diagnostic showed a gradual taper). On a smooth
signal, matching change-point *dates* across scenarios is fragile in both directions:
force too few breaks and scenarios disagree on which shifts to keep; force too many
and you match noise. So we ask two robust questions instead:

1. **Trajectory agreement** : do all scenarios trace the same shape (the U: decline
   then re-concentration)? If so, the turning points are real features of the data,
   independent of whether their exact month is pin-downable. (Spearman correlation.)
2. **Sharp-break corroboration** : give each scenario enough breaks to find all

In [3]:
print([(k, v) for k, v in globals().items()
       if re.search(r"tol|chance|break", k, re.I) and not k.startswith("_")])

[]
